In [1]:
from pyspark.sql import SparkSession

# STOP existing Spark session (safe to run even if not present)
try:
    spark.stop()
    print("Stopped existing Spark session.")
except Exception as e:
    print("No active Spark session or error stopping:", e)


pkgs = "org.apache.hadoop:hadoop-aws:3.3.4,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3"
MINIO_ENDPOINT = "http://127.0.0.1:9000"  # adjust if needed
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"

spark = (SparkSession.builder
    .appName("Iceberg Setup - Testing Timeouts")
    # keep your catalog configs if needed (omit for a quick read test)
    .config("spark.jars.packages", pkgs)
    # S3A / MinIO connection settings
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    # defensive numeric overrides
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000")
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.socket.timeout", "60000")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "10")
    .config("spark.hadoop.fs.s3a.retry.limit", "10")
    # driver/executor Java options
    .config("spark.driver.extraJavaOptions", "-Dsun.net.client.defaultReadTimeout=60000 -Dsun.net.client.defaultConnectTimeout=60000")
    .config("spark.executor.extraJavaOptions", "-Dsun.net.client.defaultReadTimeout=60000 -Dsun.net.client.defaultConnectTimeout=60000")
    .getOrCreate()
)
print("Created SparkSession:", spark)


No active Spark session or error stopping: name 'spark' is not defined
Created SparkSession: <pyspark.sql.session.SparkSession object at 0x000001DB4D07A7B0>


In [2]:
df = spark.read.csv("s3a://basetables/Sales_Invoices.csv", sep="~", header=True)
df.show(5)

+---------+----------+----------------+-------+----------------+---------------+----------------+-------------------+----------------+-----------+---------------------------+------------+----------------+--------+--------------------+----------------+-------------+-----------------+-----------+-----------+--------------------+---------------------+-------------------+------------+-------------------+
|InvoiceID|CustomerID|BillToCustomerID|OrderID|DeliveryMethodID|ContactPersonID|AccountsPersonID|SalespersonPersonID|PackedByPersonID|InvoiceDate|CustomerPurchaseOrderNumber|IsCreditNote|CreditNoteReason|Comments|DeliveryInstructions|InternalComments|TotalDryItems|TotalChillerItems|DeliveryRun|RunPosition|ReturnedDeliveryData|ConfirmedDeliveryTime|ConfirmedReceivedBy|LastEditedBy|     LastEditedWhen|
+---------+----------+----------------+-------+----------------+---------------+----------------+-------------------+----------------+-----------+---------------------------+------------+-----

In [ ]:
spark.stop()